# **Bayesian Reasoning and Decisions Under Uncertainty**

**Lecture (60 min) + lab (60 min)**  
**Setup:** Python 3; no external packages required.

By the end, you can calculate a conditional probability, apply Bayes' theorem, explain a simple Bayesian network, and make a decision that accounts for the cost of mistakes.

| Time | Activity |
|---|---|
| 0–15 min | Conditional probability and Bayes' theorem |
| 15–30 min | Base rates and diagnostic tests |
| 30–45 min | Bayesian networks and conditional independence |
| 45–60 min | Worked spam-filter example |
| 60–120 min | Lab: build and evaluate a tiny Bayesian spam decision |

## 1. Conditional probability and Bayes' theorem

Conditional probability asks how likely `A` is when we know `B` happened:

`P(A | B) = P(A and B) / P(B)`

Bayes' theorem reverses the condition:

`P(A | B) = P(B | A) * P(A) / P(B)`

`P(A)` is the **prior**, `P(B | A)` is the **likelihood**, and `P(A | B)` is the **posterior**. The prior matters: a seemingly accurate test can still produce many false alarms when the condition is rare.

In [ ]:
prevalence = 0.01
sensitivity = 0.95                 # P(positive | condition)
specificity = 0.90                 # P(negative | no condition)
false_positive_rate = 1 - specificity

p_positive = sensitivity * prevalence + false_positive_rate * (1 - prevalence)
p_condition_given_positive = sensitivity * prevalence / p_positive
print(f"Probability of condition after a positive test: {p_condition_given_positive:.1%}")

Even with 95% sensitivity, the posterior is only about 8.8% in this example because the prior is 1% and false positives are more common than true positives. Always include the population base rate and false-positive rate in diagnostic reasoning.

## 2. Bayesian networks and decisions

A **Bayesian network** is a directed acyclic graph whose nodes are random variables and whose edges encode direct probabilistic dependencies. Each node has a conditional probability table given its parents. The graph can make assumptions explicit; for example, a simple model might be `Spam -> ContainsOffer` and `Spam -> ContainsLink`.

A common simplifying assumption in a **naive Bayes** model is that features are conditionally independent given the class. It is often imperfect, but lets us combine evidence with a few probabilities. A probability alone does not determine an action: decision-making also depends on the relative cost of false positives and false negatives.

In [ ]:
p_spam = 0.20
p_offer_given_spam, p_offer_given_ham = 0.70, 0.10
p_link_given_spam, p_link_given_ham = 0.80, 0.20

# Naive Bayes: assume offer and link are independent once class is known.
spam_weight = p_spam * p_offer_given_spam * p_link_given_spam
ham_weight = (1 - p_spam) * p_offer_given_ham * p_link_given_ham
p_spam_given_both = spam_weight / (spam_weight + ham_weight)
print(f"P(spam | offer, link) = {p_spam_given_both:.1%}")

Suppose falsely sending a legitimate email to spam costs 4 units, while letting a spam message into the inbox costs 1 unit. Mark as spam when `P(spam | evidence) > cost(false positive) / (cost(false positive) + cost(false negative))`, or 80% here. The example posterior exceeds that threshold. This is a **decision threshold**, not a universal probability cutoff.

## Lab (60 min): a tiny Bayesian spam filter

Use the same simplified model with two observed features: the email contains an offer and contains a link. Assume these features are conditionally independent given whether the email is spam.

### Tasks

1. Write a reusable function that calculates `P(spam | observed features)` using Bayes' theorem. It should accept the prior and the feature likelihoods for spam and legitimate mail.
2. Calculate the posterior for an email with both features using the values below; verify it is close to 0.875.
3. Calculate the posterior for an email with an offer but no link.
4. Mark an email as spam only when doing so has lower expected cost. Use false-positive cost 4 and false-negative cost 1; calculate the decision threshold and classify both emails.
5. In two sentences, explain the conditional-independence assumption and one reason it may fail in real email.

**Suggested timing:** implement (15 min), calculate/test (15 min), decision costs (15 min), interpretation (15 min).

In [ ]:
def posterior_spam(prior, likelihoods_spam, likelihoods_ham):
    # Each likelihood list contains P(feature=value | class) for observed features.
    # TODO: multiply the prior by the class-conditional likelihoods,
    # then normalize the spam and ham weights.
    raise NotImplementedError

prior = 0.20
spam_likelihoods = {"offer": 0.70, "link": 0.80}
ham_likelihoods = {"offer": 0.10, "link": 0.20}

# Both features are present.
p_both = posterior_spam(prior, [0.70, 0.80], [0.10, 0.20])
# Offer present, link absent.
p_offer_only = posterior_spam(prior, [0.70, 1 - 0.80], [0.10, 1 - 0.20])

false_positive_cost, false_negative_cost = 4, 1
threshold = false_positive_cost / (false_positive_cost + false_negative_cost)
print(p_both, p_offer_only, threshold)

In [ ]:
# Run after implementing posterior_spam.
assert abs(p_both - 0.875) < 1e-9
assert 0 <= p_offer_only <= 1
assert threshold == 0.8
print("Bayes calculations passed.")

### Exit ticket

Submit your function, both posteriors, the threshold-based decisions, and your explanation of the model assumption. **Extension:** change the class prior to 5% and describe how the posterior changes while the likelihoods stay fixed.